# Week 8 · Notebook 2  VRAM Sizing & Model Selection

**Bytes-per-parameter math, the 7B/13B/70B sizing table, KV-cache overhead, and a picker that recommends a model + quant for your hardware.**

```
# Requirements: none (pure Python, no GPU, no downloads)
```

Part of AI Engineering Lab · ZoroLogistics case study.

## The one rule: weights = bytes/param × params

Every model-fitting decision reduces to this. Format determines bytes/param  FP32 is 4, FP16/BF16 is 2, INT8 is 1, and 4-bit (GGUF `Q4_K_M` / bitsandbytes NF4) is 0.5, i.e. roughly **1 GB per billion parameters at 4-bit**.

In [ ]:
FORMATS = {"FP32": 4.0, "FP16/BF16": 2.0, "INT8": 1.0, "INT4 / GGUF Q4": 0.5}

def weights_gb(params_b, bytes_per_param):
    return params_b * bytes_per_param

print("bytes/param table:")
for name, bpp in FORMATS.items():
    print(f"  {name:16} {bpp:>3} bytes/param -> {bpp:.1f} GB per 1B params")

## The sizing table (7B / 13B / 70B)

Weights only, before KV cache and activations  from research note 03 and knowledge-base 08.

In [ ]:
sizes = [7, 13, 70]
rows = []
for s in sizes:
    rows.append({"model": f"{s}B", "FP16_gb": weights_gb(s, 2.0),
                 "INT8_gb": weights_gb(s, 1.0), "INT4_gb": weights_gb(s, 0.5)})
import pandas as pd
print(pd.DataFrame(rows).to_string(index=False))
print("\nrule of thumb: ~1 GB per 1B params at 4-bit; ~2 GB at FP16.")

## KV-cache overhead

Weights are only the start. The **KV cache** grows with sequence length  roughly `2 × layers × heads × head_dim × seq_len × bytes/param`  and activations add ~1020% more. Leave headroom for the OS too.

In [ ]:
def kv_cache_gb(layers, heads, head_dim, seq_len, bytes_per_param=2.0):
    return 2 * layers * heads * head_dim * seq_len * bytes_per_param / 1e9

def total_footprint(params_b, bytes_per_param, layers, heads, head_dim,
                    seq_len=4096, os_reserve=2.0):
    w = weights_gb(params_b, bytes_per_param)
    kv = kv_cache_gb(layers, heads, head_dim, seq_len)
    act = 0.12 * w          # activation overhead estimate
    return w + kv + act + os_reserve

# sanity check: a 7B-class model (32 layers, 32 heads, 128 head_dim) at 4096 ctx, FP16
demo = total_footprint(7.0, 2.0, 32, 32, 128, seq_len=4096)
print(f"7B @ FP16, 4096 ctx: weights={weights_gb(7.0,2.0):.1f} + KV={kv_cache_gb(32,32,128,4096):.2f} + act + OS = {demo:.1f} GB total")

## The picker

A small model catalog with approximate architecture, a quant table, and a `pick_model(gb, task)` that returns the **largest model that fits comfortably** (20% headroom beyond the OS reserve) for the task's minimum size  applied to offline ticket triage.

In [ ]:
CATALOG = [
    {"name": "Qwen2.5 3B",   "params": 3.0,  "license": "Apache-2.0", "layers": 36, "heads": 16, "head_dim": 128},
    {"name": "Llama 3.2 3B", "params": 3.2,  "license": "Llama",      "layers": 28, "heads": 16, "head_dim": 128},
    {"name": "Qwen2.5 7B",   "params": 7.6,  "license": "Apache-2.0", "layers": 28, "heads": 28, "head_dim": 128},
    {"name": "Mistral 7B",   "params": 7.2,  "license": "Apache-2.0", "layers": 32, "heads": 32, "head_dim": 128},
    {"name": "Llama 3.1 8B", "params": 8.0,  "license": "Llama",      "layers": 32, "heads": 32, "head_dim": 128},
    {"name": "Qwen2.5 14B",  "params": 14.8, "license": "Apache-2.0", "layers": 48, "heads": 40, "head_dim": 128},
    {"name": "Llama 3.3 70B", "params": 70.0, "license": "Llama",      "layers": 80, "heads": 64, "head_dim": 128},
]
QUANTS = {"Q4_K_M": 0.5, "Q8_0": 1.0, "FP16": 2.0}
MIN_PARAMS = {"triage": 3.0, "coding": 7.0, "reasoning": 13.0}

def list_options(available_gb, task="triage", seq_len=4096, os_reserve=2.0, comfort=0.8):
    budget = available_gb * comfort
    opts = []
    for m in CATALOG:
        if m["params"] < MIN_PARAMS.get(task, 3.0):
            continue
        for qname, bpp in QUANTS.items():
            total = total_footprint(m["params"], bpp, m["layers"], m["heads"], m["head_dim"],
                                    seq_len=seq_len, os_reserve=os_reserve)
            if total <= budget:
                opts.append((m["params"], m["name"], qname, round(total, 2), m["license"]))
    opts.sort(key=lambda x: -x[0])
    return opts

def pick_model(available_gb, **kw):
    opts = list_options(available_gb, **kw)
    return opts[0] if opts else None

## Apply it: 16 GB MacBook and an 8 GB Windows laptop

Both must run **offline ticket triage**. The picker returns the largest comfortable fit; the license column is the second decision  Apache-2.0/MIT is the safe default for a commercial shipping tool, Llama carries terms you must record.

In [ ]:
for label, gb in [("16GB Apple Silicon MacBook", 16.0), ("8GB Windows laptop", 8.0)]:
    opts = list_options(gb, task="triage")
    print(f"\n{label} ({gb} GB), options that fit comfortably:")
    for p in opts[:6]:
        print(f" {p[1]:14} {p[2]:6} ~{p[3]:5} GB total ({p[4]})")
    top = pick_model(gb, task="triage")
    print(" -> recommendation:", top[1], "at", top[2])

In [ ]:
# Week 8 · Notebook 2 headline metric: estimated total GB of the recommended model for the 16GB MacBook.
mac_top = pick_model(16.0, task="triage")
print("WEEK8_NB2_RECOMMENDED_FOOTPRINT_GB:", mac_top[3])